# 第二阶段：图像识别进阶与迁移学习 (猫狗分类)

恭喜完成 MNIST 分类任务！MNIST 的图片是完美的 28x28 灰度图，但在真实的科研与工程中，图片往往大小不一、色彩丰富（RGB 三通道）、且数量极其有限。

本节课我们将解锁深度学习最强杀招——**迁移学习（Transfer Learning）**。我们将加载微软训练好的 ResNet18 模型“大脑”，直接应用到我们的猫狗分类任务中，这也就是所谓地“站在巨人的肩膀上”。

同样，我们将严格遵循 **7步深度教学法** 与 **11步全链路流程**。

## 前置准备：模拟真实数据集生成

在正式开始前，为了让你这篇代码可以直接跑通，我们写几行代码在本地自动生成一个极小型的“虚拟猫狗数据集（随机噪点图）”文件夹结构。

In [1]:
import os
import numpy as np
from PIL import Image

# 自动生成假的猫狗数据集目录，用于演示 ImageFolder 读取真实文件的流程
data_dir = "./fake_catdog_data"
for split in ['train', 'val']:
    for class_name in ['cats', 'dogs']:
        os.makedirs(os.path.join(data_dir, split, class_name), exist_ok=True)
        # 每个类别生成 5 张随机噪点图片 (模拟真实 jpg)
        for i in range(5):
            img_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
            img = Image.fromarray(img_array)
            img.save(os.path.join(data_dir, split, class_name, f"{class_name}_{i}.jpg"))

print("已成功在当前目录生成模拟的 猫狗图片 数据集文件夹: ./fake_catdog_data")

已成功在当前目录生成模拟的 猫狗图片 数据集文件夹: ./fake_catdog_data


## 步骤 1-3：数据增强 ➔ Dataset ➔ DataLoader

### ① 知识讲解
- **数据增强 (Data Augmentation)**：对图片进行随机裁剪、旋转、翻转。让模型每次看到的猫都稍微不一样。
- **ImageFolder**：PyTorch 内置的读取自定义图片的神器。只要你的文件夹按照 `类名/图片.jpg` 组织，它就能自动打上标签（0, 1, 2...）。

### ② 为什么这么做
深度学习模型非常容易**过拟合（Overfitting）**，也就是“死记硬背”。通过数据增强，相当于凭空创造了几十倍的数据量，逼迫模型去学习真正的特征（猫耳朵、狗鼻子），而不是死记背景颜色。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# ③ 完整代码 & ④ 代码逐行解释

# 1. 定义数据增强与预处理流程
# 训练集：需要加上疯狂的随机变换
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),     # 随机裁剪并强制缩放到 224x224 (模型标准的输入大小)
    transforms.RandomHorizontalFlip(),     # 50% 概率水平翻转图片
    transforms.ToTensor(),                 # 必须：转为 Tensor 并缩放至 [0, 1]
    # ImageNet 经典标准化参数 (所有用 torchvision 预训练模型的人，都必须用这组魔法数字)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集/测试集：绝对不能增加随机变换！只能老老实实缩放和裁剪居中
val_transforms = transforms.Compose([
    transforms.Resize(256),                # 先放大一点
    transforms.CenterCrop(224),            # 取正中心最清晰的 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. ImageFolder 自动加载自定义图片数据集
train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=val_transforms)

# 查看自动分配的标签映射表
print("自动生成的类别映射字典:", train_dataset.class_to_idx) # 输出类似: {'cats': 0, 'dogs': 1}

# 3. DataLoader 打包
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

自动生成的类别映射字典: {'cats': 0, 'dogs': 1}


### 💡 深度避坑与实践建议
- **⑤ 容易踩坑**：验证集（val/test）千万不能加 `RandomHorizontalFlip` 这类随机操作！此外，只要你使用了官方的预训练权重，你的 `Normalize` 参数就必须死记硬背那组魔法数字：均值 `[0.485, 0.456, 0.406]`，方差 `[0.229, 0.224, 0.225]`。
- **⑥ 科研实验室**：真实的科研数据集往往没有完美的文件夹结构。我们通常会手写一个继承自 `torch.utils.data.Dataset` 的类，读取一个 `labels.csv`，在里面的 `__getitem__` 方法中用 `PIL.Image.open()` 或 `cv2.imread()` 读取路径图片。
- **⑦ 工程实践建议**：工业级数据增强早就不用 `torchvision.transforms` 了，而是用 `Albumentations` 库。它是用 C++ 写底层，处理图像速度极快，是 Kaggle 比赛和企业生产的标配。

## 步骤 4：模型搭建 (迁移学习微调 ResNet)

### ① 知识讲解
**迁移学习 (Transfer Learning)**：直接下载大厂在几百万张图片 (ImageNet) 上训练好的深度残差网络 (ResNet)，它的前几层已经完美学会了如何识别“边缘、纹理、猫耳朵狗鼻子”。我们只需要把最后一层负责分类的全连接层（1000分类），砍掉换成我们自己的（2分类），然后训练这最后一层即可。

### ② 为什么这么做
自己从零（随机初始化）去训练一个深层网络需要几万张图片和好几周的显卡算力。迁移学习只需要几百张图，训练几分钟，准确率就能碾压从零训练的模型。

In [3]:
# ③ 完整代码 & ④ 代码逐行解释
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# 1. 下载预训练模型 (weights=models.ResNet18_Weights.DEFAULT 代表加载最好的那批权重)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. [可选] 冻结网络的所有特征提取层 (即不更新这些层的梯度，大大节省显存和加速训练)
for param in model.parameters():
    param.requires_grad = False

# 3. 替换最后一层
# resnet18 的最后一层叫 fc (全连接层)
num_ftrs = model.fc.in_features # 获取最后一层原本的输入维度 (通常是 512)
# 创建一个新的全连接层，输入依然是 512，输出改为我们的 2 分类 (猫、狗)
# 新创建的层，默认 requires_grad=True，所以整个网络现在只有这层在被训练
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)
print("预训练 ResNet18 模型已加载，最后一层已修改为 2 分类。")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/zoupray/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 8.59MB/s]


预训练 ResNet18 模型已加载，最后一层已修改为 2 分类。


### 💡 深度避坑与实践建议
- **⑤ 容易踩坑**：不同模型的最后一层名字不一样。ResNet 叫 `model.fc`，VGG 或 DenseNet 叫 `model.classifier`，Swin Transformer 叫 `model.head`。替换前可以用 `print(model)` 查看它的底层结构名字。
- **⑥ 科研实验室**：冻结网络（Frozen Backbone）通常只用于数据极少或显卡极差的情况。在科研打榜刷 SOTA 时，通常是**全部层放开训练（Full Fine-Tuning）**，但会给骨干网络（Backbone）设置一个非常小的学习率（如 `1e-5`），而给新加的全连接分类器设置大一点的学习率（如 `1e-3`）。
- **⑦ 工程实践建议**：工业界常用的主干网络已经从 ResNet 转为了 `ConvNeXt`，或者轻量级的 `MobileNetv3`，以及能够兼顾全局注意力的视觉 Transformer（如 `Swin-Transformer`）。

## 步骤 5-11：全链路闭环 (训练、保存与 ONNX 导出)

训练循环与之前高度一致，这里我们快速把全链路闭环跑通。

In [5]:
# 步骤 5-6 训练与验证
criterion = nn.CrossEntropyLoss()
# 注意：因为我们上面冻结了特征层，所以扔给优化器的只有 model.fc 的参数
optimizer = optim.AdamW(model.fc.parameters(), lr=0.001)

epochs = 20  # 增加训练轮数，让模型充分学习
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # 简易验证阶段
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    val_acc = 100. * correct / len(val_dataset)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | 验证准确率: {val_acc:.2f}%")


# 步骤 7-10 保存、加载与推理
torch.save(model.state_dict(), "catdog_resnet18.pth")
print("\n模型权重已保存！")

model.load_state_dict(torch.load("catdog_resnet18.pth"))
model.eval()

# 取出验证集的第一张图片进行推理
img_tensor, label = val_dataset[0]
with torch.no_grad():
    out = model(img_tensor.unsqueeze(0).to(device))
    pred_idx = out.argmax(dim=1).item()

# 获取 idx 对应的类名字符串 (0 -> 'cats', 1 -> 'dogs')
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
print(f"单张图片推理结果: 模型预测为 -> {idx_to_class[pred_idx]}, 真实标签为 -> {idx_to_class[label]}")


# 步骤 11 ONNX 导出
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "catdog_resnet18.onnx"
torch.onnx.export(
    model, dummy_input, onnx_path,
    export_params=True, opset_version=11,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"模型已成功导出为 ONNX，可以直接提供给 C++/TensorRT 进行工业级部署: {onnx_path}")

Epoch 1/20 | Train Loss: 2.7759 | 验证准确率: 50.00%
Epoch 2/20 | Train Loss: 1.7123 | 验证准确率: 50.00%
Epoch 3/20 | Train Loss: 2.3024 | 验证准确率: 50.00%
Epoch 4/20 | Train Loss: 1.7396 | 验证准确率: 50.00%
Epoch 5/20 | Train Loss: 2.3057 | 验证准确率: 50.00%
Epoch 6/20 | Train Loss: 2.0069 | 验证准确率: 50.00%
Epoch 7/20 | Train Loss: 2.0073 | 验证准确率: 50.00%
Epoch 8/20 | Train Loss: 2.1388 | 验证准确率: 60.00%
Epoch 9/20 | Train Loss: 2.1102 | 验证准确率: 70.00%
Epoch 10/20 | Train Loss: 1.8046 | 验证准确率: 60.00%
Epoch 11/20 | Train Loss: 2.0084 | 验证准确率: 60.00%
Epoch 12/20 | Train Loss: 1.8174 | 验证准确率: 60.00%
Epoch 13/20 | Train Loss: 2.3964 | 验证准确率: 50.00%
Epoch 14/20 | Train Loss: 1.7975 | 验证准确率: 60.00%
Epoch 15/20 | Train Loss: 1.8570 | 验证准确率: 60.00%
Epoch 16/20 | Train Loss: 2.0524 | 验证准确率: 50.00%
Epoch 17/20 | Train Loss: 2.3100 | 验证准确率: 60.00%
Epoch 18/20 | Train Loss: 1.8085 | 验证准确率: 60.00%
Epoch 19/20 | Train Loss: 2.1731 | 验证准确率: 70.00%


/var/folders/4x/gykrz2md4nz_j7l5r35_m6880000gn/T/ipykernel_29703/3581514839.py:52: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0716 18:44:13.725000 29703 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Epoch 20/20 | Train Loss: 1.7233 | 验证准确率: 70.00%

模型权重已保存！
单张图片推理结果: 模型预测为 -> dogs, 真实标签为 -> cats
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/ve

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
模型已成功导出为 ONNX，可以直接提供给 C++/TensorRT 进行工业级部署: catdog_resnet18.onnx


## 🏆 第二阶段结语

走到这里，你已经掌握了工业界图像分类的**终极方案**：无论遇到多奇葩的自定义数据集，只需要用 `ImageFolder` 或自定义 `Dataset` 读入，加上疯狂的数据增强（Transforms），然后挂上一个大厂预训练好的 ResNet/ConvNeXt，替换掉最后一层，冻结或微调它。

在真实的猫狗万张图片数据集中，这套代码仅需几分钟就能跑到 98% 以上的惊人准确率！

请运行本代码并观察输出。下一阶段，我们将踏入真正科研刚需的深水区——**目标检测（YOLO，不仅要分出猫狗，还要画出框）**！